# Othello — Training and evaluation of agents

## 1 — Imports

In [ ]:
import sys
sys.path.insert(0, ".")          # adjust to your project path

import numpy as np
import matplotlib.pyplot as plt

from train              import train
from train_multi_agent  import train_multi_agent
from evaluation         import evaluate_fair, play_game

from agents.random_agent    import RandomAgent
from agents.greedy_agent    import GreedyAgent
from agents.heuristic_agent import HeuristicAgent
from agents.minimax_agent   import MinimaxAgent

OPPONENT_CLASSES = {
    "random":    RandomAgent,
    "greedy":    GreedyAgent,
    "heuristic": HeuristicAgent,
    "minimax":   MinimaxAgent,
}

print("Imports OK")

Importy OK


## 2 — Configuration

In [2]:
BOARD_SIZE    = 6
NUM_EPISODES  = 5_000
MODELS_DIR    = "models"

## 3 — Training agents

Each `train()` call saves the model to disk and returns a dictionary with history.  
Comment out agents you don't want to train.

In [3]:
# Classic DQN
result_dqn = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = False,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/dqn.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_dqn = result_dqn["agent"]


Training: dqn | hw=0.0 | per=False | opponent=mix | episodes=5000
[   500/5000] opp=heuristic  ε=0.606 | avg_r=0.030 win%=0.500 | loss=0.0072 | W/D/L=227/18/255


KeyboardInterrupt: 

In [ ]:
# Guided DQN  (heuristic_weight > 0)
result_guided = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = False,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/guided_dqn.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided = result_guided["agent"]


In [ ]:
# PER DQN  (prioritized replay)
result_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = True,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/per_dqn.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_per = result_per["agent"]


In [ ]:
# Guided + PER
result_guided_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = True,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/guided_per_dqn.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided_per = result_guided_per["agent"]


## 4 — Training curves

In [ ]:
def moving_avg(arr, w=100):
    return np.convolve(arr, np.ones(w)/w, mode="valid") if len(arr) >= w else np.array(arr)

results = {
    "DQN":        result_dqn,
    "Guided DQN": result_guided,
    "PER DQN":    result_per,
    "Guided PER": result_guided_per,
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, res in results.items():
    axes[0].plot(moving_avg(res["wins"],  100), label=name)
    axes[1].plot(moving_avg(res["losses"], 50), label=name)

axes[0].set_title("Win rate MA-100")
axes[0].set_xlabel("episode")
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].legend()

axes[1].set_title("Loss MA-50")
axes[1].set_xlabel("train step")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5 — Evaluation vs. rule-based agents

In [ ]:
agents = {
    "DQN":        agent_dqn,
    "Guided DQN": agent_guided,
    "PER DQN":    agent_per,
    "Guided PER": agent_guided_per,
}

print(f"{'Agent':<16}", end="")
for opp in OPPONENT_CLASSES:
    print(f"  {opp:>10}", end="")
print(f"  {'average':>8}")
print("-" * 65)

eval_scores = {}
for agent_name, agent in agents.items():
    agent.q_net.eval()
    row = {}
    for opp_name, opp_cls in OPPONENT_CLASSES.items():
        res = evaluate_fair(agent_a=agent, agent_b_class=opp_cls,
                            board_size=BOARD_SIZE, n_games=100)
        row[opp_name] = res["score"]
    eval_scores[agent_name] = row
    agent.q_net.train()

    scores = list(row.values())
    print(f"{agent_name:<16}", end="")
    for s in scores:
        print(f"  {s:>10.3f}", end="")
    print(f"  {np.mean(scores):>8.3f}")

## 6 — Tournament between DQN agents (round-robin)

In [ ]:
from environment import OthelloEnv

N_GAMES = 100   # games per pair

names = list(agents.keys())
n     = len(names)
matrix = np.zeros((n, n))

for i, name_a in enumerate(names):
    for j, name_b in enumerate(names):
        if i == j:
            matrix[i, j] = 0.5
            continue

        ag_a = agents[name_a]
        ag_b = agents[name_b]
        ag_a.q_net.eval()
        ag_b.q_net.eval()

        a_wins = draws = b_wins = 0
        for game_idx in range(N_GAMES):
            role_a = 1 if game_idx % 2 == 0 else -1
            env = OthelloEnv(board_size=BOARD_SIZE)
            obs = env.reset()
            done = False
            while not done:
                if env.current_player == role_a:
                    action = ag_a.select_action(obs)
                else:
                    action = ag_b.select_action(obs)
                obs, _, done, info = env.step(action)
            w = info["winner"]
            if   w == role_a:  a_wins += 1
            elif w == -role_a: b_wins += 1
            else:              draws  += 1

        matrix[i, j] = (a_wins + 0.5 * draws) / N_GAMES

print(f"{'':16}", end="")
for name in names:
    print(f"  {name:>14}", end="")
print()
print("-" * (16 + 16 * n))

for i, name in enumerate(names):
    print(f"{name:<16}", end="")
    for j in range(n):
        print(f"  {matrix[i,j]:>14.3f}", end="")
    print()

In [ ]:
# Ranking by average score
avg = matrix.mean(axis=1)
ranking = np.argsort(avg)[::-1]

print("\nFinal ranking:")
for rank, idx in enumerate(ranking, 1):
    print(f"  {rank}. {names[idx]:<16}  average score = {avg[idx]:.3f}")

## 7 — Multi-agent training (optional)

Alternative to section 3 — all agents train simultaneously,
each episode a learner and opponent are randomly selected.

In [ ]:
trained_agents, stats = train_multi_agent(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    model_dir     = f"{MODELS_DIR}/multi",
    print_every   = 500,
)

print("\nStatistics after multi-agent training:")
for name, s in stats.items():
    print(f"  {name:<14}  W/D/L = {s['wins']}/{s['draws']}/{s['losses']}")

NameError: name 'train_multi_agent' is not defined

## 8 — Loading saved model and StudentAgent

In [ ]:
from agent         import DQNAgent
from student_agent import StudentAgent

# Load via DQNAgent
loaded = DQNAgent(board_size=BOARD_SIZE)
loaded.load(f"{MODELS_DIR}/guided_per_dqn.pth")
loaded.q_net.eval()
print("DQNAgent loaded")

# Load via StudentAgent (submission format)
student = StudentAgent(
    board_size      = BOARD_SIZE,
    checkpoint_path = f"{MODELS_DIR}/guided_per_dqn.pth",
)
print("StudentAgent loaded")

# Verification — StudentAgent must always return a legal action
from environment import OthelloEnv
env = OthelloEnv(board_size=BOARD_SIZE)
obs = env.reset()
action = student.select_action(obs)
assert action in obs["legal_actions"]
print(f"select_action → {action}  (legal: {obs['legal_actions']})  ✓")